In [ ]:
import requests
import lxml.html
from collections import defaultdict
import pandas as pd

## State Court Data Ingestion
Exploration of data sources for state courts for ingestion into Courts table. Working code now lives in ingest_courts_data.py.

Discussion: I think goal of functions in ingestion should be to produce an output which can be easily loaded into Django - not totally sure what the best format for that is, I think bulk create method takes a dict or we could go data -> pandas dataframe -> pandas to django convenience methods -> django. Should decide on a common output for ingestion functions.

### NCSC State Court Info
Scrape data on court selection methods, term lengths, etc for state courts from archived NCSC site: http://web.archive.org/web/20151022190427/http://www.judicialselection.us/judicial_selection/methods/selection_of_judges.cfm?state=

In [ ]:
# request and parse html
archive_ncsc_url = "http://web.archive.org/web/20151022190427/http://www.judicialselection.us/judicial_selection/methods/selection_of_judges.cfm?state="
r = requests.get(archive_ncsc_url)
response = lxml.html.fromstring(r.text)

In [ ]:
# parse html for state court info
state_data_dict = defaultdict(lambda: defaultdict(dict))
other_thing = []

for item in response.xpath('//div[@id="content"]/*'):
    if item.tag in ["h2", "h3", "p"]:
        continue
    # update the state
    elif item.tag == "div" and "yellow_box" in item.classes:
        state_str = item.xpath("h4/text()")[0]
    # get data for one state
    elif item.tag == "table":
        state_data_str = item.text_content()
        state_data_list = [
            item for item in state_data_str.replace("\t", "").split("\n") if item != ""
        ]
        stat_key = state_data_list[0]

        i = 1
        while i < len(state_data_list) - 1:
            court = state_data_list[i]
            stat_value = state_data_list[i + 1]
            state_data_dict[state_str][court][stat_key] = stat_value
            i += 2

state_data_dict

In [ ]:
# variable names improved in real code
# reformat to flat dictionaries and convert to dataframe
things = []
for state, court_dict in state_data_dict.items():
    for court, info in court_dict.items():
        flat = {"State": state, "Court": court, **info}
        things.append(flat)
df = pd.DataFrame(things)
df

In [ ]:
selection_methods = (
    df["Method of Selection (full term)"].str.replace("*", "").str.replace("+", "").unique()
)
selection_methods

### CourtListener
Get list of courts from CourtListener's dedicated courts API: https://www.courtlistener.com/api/rest/v4/courts/

API token required! Just need to sign in on CourtListener site and find token under developer tools.

In [ ]:
base_url = "https://www.courtlistener.com/"
courts_url = "api/rest/v4/courts/"
API_token = "REPLACE WITH YOUR API TOKEN"

In [ ]:
courts_request = requests.get(
    base_url + courts_url, headers={"Authorization": f"Token {API_token}"}
)
courts_request.status_code

In [ ]:
courts_results = []
page_url = base_url + courts_url
while page_url is not None:
    r = requests.get(page_url, headers={"Authorization": f"Token {API_token}"})
    print(page_url, r.status_code)
    response = r.json()
    results = response["results"]
    courts_results.extend(results)
    page_url = response["next"]

In [ ]:
courtlistener_df = pd.DataFrame(courts_results)
courtlistener_df

Lots of the variables are kind of confusing; some jurisdictions mislabeled etc. Their state id decisions are crazy (Alabama is ala, Alaska is alaska, etc. wtf)

In [ ]:
# filter to jurisdiction "S" for statewide courts and non-NaN start date,
# NaN end date to get state supreme courts only.
cl_ssc_df = courtlistener_df[
    (courtlistener_df["jurisdiction"] == "S")
    & (courtlistener_df["start_date"].notna())
    & (courtlistener_df["end_date"].isna())
]
cl_ssc_df

In [ ]:
# save to csv file
# courtlistener_df.to_csv("courts_courtlistener.csv")